# Tome 4, Chapter 9: Complex analysis for computing (Python companion)

Exercises first, solutions at the end. Cauchy's formula becomes a numerical method: Taylor coefficients by FFT, zeros counted by an integral, sums and integrals computed from residues, and a root finder that needs no starting guess. Requirements: numpy, sympy and mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P9.1 (★★) Taylor coefficients by FFT.** By Theorem 9.11, $c_n=\frac1{2\pi i}\int_{C(0,r)}\frac{f(w)}{w^{n+1}}\,\mathrm dw$. Approximate this integral with the trapezoid rule on $N$ equally spaced points of the circle, and observe that all $N$ coefficients at once are a single FFT. Use it to recover the Bernoulli numbers $B_0,\dots,B_{20}$ from $\frac z{e^z-1}$ with $N=64$, for the radii $r=1$, $3$ and $6$. Which radius works, and why do the other two fail? (Recall from Corollary 9.12 that the radius of convergence is $2\pi$.) Then measure how fast the trapezoid rule converges for $\frac1{2\pi i}\int_{C(0,1)}\frac{e^z}z\,\mathrm dz=1$.

**P9.2 (★) Counting zeros.** Compute $\frac1{2\pi i}\int\frac{p'}{p}$ numerically on the circles $|z|=1$, $1.5$, $2$ for $p=z^5+3z+1$ and $p=z^4-6z+3$, and compare with Theorem 9.38, the example after it, Exercise 9.4 and `numpy.roots`. Then compute the winding number of the image of the unit circle (Figure 9.3) by following the angle of $p(e^{it})$.

**P9.3 (★) Integrals by residues, checked.** Check Exercise 9.1 for several $n$, Example 9.30 for $a=0.3$, and the Fresnel integral of Exercise 9.2 by quadrature to 20 digits.

**P9.4 (★★) Sums by residues, automatically.** Write a function that, given a rational function $\frac{P}{Q}$ with $\deg Q\ge\deg P+2$ and simple non-integer poles, returns $\sum_{n\in\Z}\frac{P(n)}{Q(n)}$ by Theorem 9.33, finding the poles numerically. Compare with direct summation for $\frac1{n^2+1}$, $\frac1{n^4+1}$ and $\frac{n^2+n}{n^6+3}$.

**P9.5 (★) The Gamma function off the real line.** Check Theorem 9.36: $|\Gamma(\frac12+iy)|^2=\frac\pi{\cosh\pi y}$, the reflection formula at a non-real point, and the residue at $-3$.

**P9.6 (★★★) CS application: a root finder without starting guesses.** Newton's method needs a good starting point, and clusters of close roots confuse it. Write a recursive *quadtree* algorithm: count the zeros of $p$ inside a square with the argument principle (Theorem 9.37); if there are none, discard the square; if there is one, polish it with Newton from the centre; otherwise split the square into four and recurse. Test it on a degree-6 polynomial with two roots at distance $0.01$.

## Solutions

### P9.1

In [ ]:
import numpy as np
from math import factorial
from fractions import Fraction
import sympy as sp
def taylor_by_fft(f, r, N):
    """First N Taylor coefficients of f at 0 by Cauchy's formula on the circle |z| = r, trapezoid rule with N nodes."""
    z = r * np.exp(2j * np.pi * np.arange(N) / N)
    return np.fft.fft(f(z)) / N / r ** np.arange(N)
f = lambda z: z / np.expm1(z)
exact = [sp.bernoulli(n) / factorial(n) for n in range(21)]
exact[1] = sp.Rational(-1, 2)          # convention B_1 = -1/2 for z/(e^z - 1)
for r in (1.0, 3.0, 6.0):
    c = taylor_by_fft(f, r, 64)
    err = max(abs(c[n] - float(exact[n])) * factorial(n) for n in range(21))
    print(f"r = {r}: max error on B_0..B_20 = {err:.1e}")
c = taylor_by_fft(f, 3.0, 64)
print("B_12 recovered:", (c[12] * factorial(12)).real, " exact:", float(sp.bernoulli(12)), "=", sp.bernoulli(12))

In [ ]:
import mpmath as mp
g = lambda z: np.exp(z)
for N in (4, 8, 12, 16, 20):
    approx = np.mean(g(np.exp(2j * np.pi * np.arange(N) / N)))      # (1/2 pi i) int e^z/z dz = e^0 = 1
    print(f"N = {N:2d} nodes: error of the trapezoid rule for (1/2 pi i) int e^z dz/z = {abs(approx - 1):.1e}")

The function $z\mapsto f(re^{2\pi ik/N})$, $k=0,\dots,N-1$, has DFT $\sum_kf(re^{2\pi ik/N})e^{-2\pi ikn/N}$, and dividing by $Nr^n$ gives the trapezoid approximation of $c_n$: that is the whole method. With $r=3$ the Bernoulli numbers come out with absolute error about $10^{-8}$, and $B_{12}=-\frac{691}{2730}$ is recognisable. The two failures have opposite causes. The trapezoid rule actually computes $c_n+c_{n+N}r^N+c_{n+2N}r^{2N}+\cdots$ (aliasing, Chapter 7); since $c_k$ decays like $(2\pi)^{-k}$, this is harmless when $r$ is well inside the radius $2\pi$, but at $r=6$ the factor $(6/2\pi)^{64}\approx0.05$ ruins it. At $r=1$ the aliasing is negligible, but the rounding errors of order $10^{-16}$ in the samples are divided by $r^n$ and then multiplied by $n!$ to get $B_n$; for $n=20$, $20!\approx2.4\cdot10^{18}$. The best radius balances the two.

The second experiment shows why this works so well: for a function holomorphic on a neighbourhood of the circle, the trapezoid rule converges *geometrically*. Twenty points give the integral to machine precision. The reason is the same aliasing formula: the error is a sum of Taylor coefficients of index $\ge N$.

### P9.2

In [ ]:
def count_zeros(p, dp, center, radius, N=2000):
    """(1/2 pi i) int p'/p over the circle, trapezoid rule."""
    z = center + radius * np.exp(2j * np.pi * np.arange(N) / N)
    dz = 2j * np.pi * (z - center) / N
    return (np.sum(dp(z) / p(z) * dz) / (2j * np.pi))
P = np.poly1d([1, 0, 0, 0, 3, 1]); dP = P.deriv()
Q = np.poly1d([1, 0, 0, -6, 3]); dQ = Q.deriv()
for name, p, dp in (("z^5+3z+1", P, dP), ("z^4-6z+3", Q, dQ)):
    for r in (1.0, 1.5, 2.0):
        n = count_zeros(p, dp, 0, r)
        print(f"{name}: zeros in |z|<{r}: integral = {n.real:+.10f}{n.imag:+.1e}i, numpy roots: {np.sum(np.abs(p.roots) < r)}")
w = P(np.exp(2j * np.pi * np.arange(4001) / 4000))
print("winding of the image of |z|=1 around 0, from the unwrapped angle:", (np.unwrap(np.angle(w))[-1] - np.angle(w[0])) / (2 * np.pi))

The integrals are integers to ten digits, as they must be, and match the root counts: one zero of $z^5+3z+1$ in the unit disc and five in $|z|<1.5$; one zero of $z^4-6z+3$ in the unit disc, none more up to $1.5$ (the other three have modulus between $1.5$ and $2$). A computed value like $2.9999$ simply rounds to $3$: because the answer is an integer, a modest accuracy suffices. Danger comes only when a zero is very close to the contour, where $p'/p$ is huge and the trapezoid rule needs many more points. The unwrapped angle of $p(e^{it})$ gains exactly $2\pi$: the image curve winds once around $0$, as in Figure 9.3.

### P9.3

In [ ]:
mp.mp.dps = 30
for n in (2, 3, 4, 5, 10):
    num = mp.quad(lambda x: 1 / (1 + x ** n), [0, 1, mp.inf])
    print(f"n = {n:2d}: quadrature {mp.nstr(num, 20)}   (pi/n)/sin(pi/n) = {mp.nstr(mp.pi / n / mp.sin(mp.pi / n), 20)}")
a = mp.mpf("0.3")
num = mp.quad(lambda t: mp.exp(a * t) / (1 + mp.exp(t)), [-mp.inf, 0, mp.inf])
print("int e^{at}/(1+e^t), a = 0.3:", mp.nstr(num, 20), "  pi/sin(pi a):", mp.nstr(mp.pi / mp.sin(mp.pi * a), 20))
fc = mp.quadosc(lambda x: mp.cos(x ** 2), [0, mp.inf], zeros=lambda n: mp.sqrt(mp.pi * (n - mp.mpf(1) / 2)))
print("int cos(x^2):", mp.nstr(fc, 20), "  sqrt(pi/8):", mp.nstr(mp.sqrt(mp.pi / 8), 20))

All agree to $20$ digits. For $n=2$ the integral is $\frac\pi2$, and as $n\to\infty$ it tends to $1$, the integral of the indicator of $[0,1]$. The Fresnel integral converges only because of cancellations; `quadosc` integrates between the zeros $\sqrt{\pi(k-\frac12)}$ of $\cos x^2$ and accelerates the resulting alternating series.

### P9.4

In [ ]:
def residue_sum(num, den):
    """Sum over n in Z of num(n)/den(n): minus the residues of pi cot(pi z) num(z)/den(z) at the zeros of den (assumed simple)."""
    dden = [c * (len(den) - 1 - i) for i, c in enumerate(den[:-1])]     # coefficients of den'
    total = 0
    for a in mp.polyroots(den, maxsteps=200, extraprec=200):
        total -= mp.pi * mp.cot(mp.pi * a) * mp.polyval(num, a) / mp.polyval(dden, a)
    return total
tests = [("1/(n^2+1)", [1], [1, 0, 1]), ("1/(n^4+1)", [1], [1, 0, 0, 0, 1]), ("(n^2+n)/(n^6+3)", [1, 1, 0], [1, 0, 0, 0, 0, 0, 3])]
for name, num, den in tests:
    direct = mp.nsum(lambda n: mp.polyval(num, n) / mp.polyval(den, n), [-mp.inf, mp.inf])
    res = residue_sum(num, den)
    print(f"sum of {name:16s}: residues {mp.nstr(mp.re(res), 22)}  direct {mp.nstr(direct, 22)}  |imaginary part| {mp.nstr(abs(mp.im(res)), 2)}")

For $\frac1{n^2+1}$ the residue sum is $\pi\coth\pi$ (Exercise 9.3). For the other two no one would look up a closed form, yet the method gives $22$ digits from two to six residues, while direct summation needs the convergence accelerators of Chapter 3. This is a practical algorithm, and the standard route to closed forms for such sums. The imaginary parts cancel because the poles come in conjugate pairs.

### P9.5

In [ ]:
for y in (0, 1, 3, 10):
    lhs = abs(mp.gamma(mp.mpf(1) / 2 + 1j * y)) ** 2
    print(f"y = {y:2d}: |Gamma(1/2+iy)|^2 = {mp.nstr(lhs, 18):>24s}   pi/cosh(pi y) = {mp.nstr(mp.pi / mp.cosh(mp.pi * y), 18)}")
z = mp.mpc("0.3", "2.1")
print("reflection at z = 0.3+2.1i:", mp.nstr(mp.gamma(z) * mp.gamma(1 - z), 15), " vs ", mp.nstr(mp.pi / mp.sin(mp.pi * z), 15))
print("residue of Gamma at -3:", mp.nstr(mp.limit(lambda e: e * mp.gamma(-3 + e), 0), 15), "  (-1)^3/3! =", -1 / 6)

On the critical line $\operatorname{Re}z=\frac12$, $|\Gamma|$ decays like $\sqrt{2\pi}e^{-\pi|y|/2}$: at $y=10$ it is already $10^{-7}$. This exponential decay is why the factor $\Gamma(\frac s2)$ in Chapter 10 makes the completed zeta function so well-behaved.

### P9.6

In [ ]:
def roots_by_argument(p, dp, center, half, tol=1e-6, out=None, depth=0):
    """Quadtree root finder: count the zeros in a square with the argument principle, split until each square holds one, then Newton."""
    if out is None:
        out = []
    corners = [center + half * c for c in (1 - 1j, 1 + 1j, -1 + 1j, -1 - 1j, 1 - 1j)]
    total = 0
    for a, b in zip(corners, corners[1:]):
        t = (np.arange(400) + 0.5) / 400                            # midpoint rule on each side
        z = a + (b - a) * t
        total += np.sum(dp(z) / p(z)) * (b - a) / 400
    n = int(round((total / (2j * np.pi)).real))
    if n == 0:
        return out
    if n == 1 or half < tol:
        z = center
        for _ in range(50):
            z = z - p(z) / dp(z)
        out.append((z, n, depth))
        return out
    for c in (1 + 1j, 1 - 1j, -1 + 1j, -1 - 1j):
        roots_by_argument(p, dp, center + c * half / 2, half / 2, tol, out, depth + 1)
    return out
R = np.poly1d(np.poly([0.3, 0.31, -1 + 1j, -1 - 1j, 2.2j, 1.5]))     # two close roots
dR = R.deriv()
found = roots_by_argument(R, dR, 0.013 + 0.007j, 4.0)
for z, n, d in sorted(found, key=lambda t: (t[0].real, t[0].imag)):
    print(f"root {z.real:+.12f}{z.imag:+.12f}i  isolated at depth {d}")
print("numpy roots:", np.sort_complex(R.roots).round(12))

Every root is found, including the pair $0.30$, $0.31$, which the quadtree separates at depth $9$, where the squares have side about $0.016$. Isolated roots are found at depth $1$ to $3$. The integer returned by the argument principle is robust (we round it), so the algorithm needs no starting guess and cannot miss a root inside the initial square. Its weak points are roots lying on or very near a square's edge (where $p'/p$ is large; a careful implementation shifts the grid) and multiple roots (which never separate, hence the depth limit). Professional root finders combine such counting with eigenvalue methods: `numpy.roots` computes the eigenvalues of the companion matrix.